In [ ]:
# rock_paper_scissors_qlearning.py
import random
from collections import defaultdict

ACTIONS = ["rock", "paper", "scissors"]
BEATS = {"rock": "scissors", "paper": "rock", "scissors": "paper"}

def result(my, opp):
    if my == opp: return 0
    return 1 if BEATS[my] == opp else -1

class QAgent:
    def __init__(self, alpha=0.1, gamma=0.9, eps=0.2, eps_min=0.02, eps_decay=0.995):
        self.Q = defaultdict(lambda: {a: 0.0 for a in ACTIONS})
        self.alpha, self.gamma = alpha, gamma
        self.eps, self.eps_min, self.eps_decay = eps, eps_min, eps_decay

    def act(self, state):
        # ε-greedy
        if random.random() < self.eps:
            return random.choice(ACTIONS)
        q = self.Q[state]
        return max(q, key=q.get)

    def learn(self, s, a, r, s2):
        qsa = self.Q[s][a]
        max_next = max(self.Q[s2].values())
        self.Q[s][a] = qsa + self.alpha * (r + self.gamma * max_next - qsa)

    def decay(self):
        self.eps = max(self.eps_min, self.eps * self.eps_decay)

def human_to_action(inp):
    return {"r": "rock", "p": "paper", "s": "scissors"}.get(inp)

def main(rounds=50):
    print("Rock-Paper-Scissors (Q-learning). 輸入 r/p/s；Enter 讓隨機玩家代打。")
    agent = QAgent()
    score = 0
    prev_opp = None   # 狀態 = 對手上一手

    for t in range(1, rounds + 1):
        # 讀玩家或隨機
        s = input(f"\nRound {t} 你的出拳(r/p/s/空白隨機): ").strip().lower()
        opp = human_to_action(s) or random.choice(ACTIONS)

        state = prev_opp  # 依據你上一手來決策
        a = agent.act(state)
        r = result(a, opp)
        score += r

        next_state = opp
        agent.learn(state, a, r, next_state)
        agent.decay()
        prev_opp = opp

        outcome = "AI贏" if r == 1 else ("平手" if r == 0 else "AI輸")
        print(f"你: {opp:9s} ｜ AI: {a:9s} ｜ 結果: {outcome} ｜ 累積分數: {score} ｜ ε={agent.eps:.3f}")

if __name__ == "__main__":
    main()


Rock-Paper-Scissors (Q-learning). 輸入 r/p/s；Enter 讓隨機玩家代打。



Round 1 你的出拳(r/p/s/空白隨機):  r


你: rock      ｜ AI: rock      ｜ 結果: 平手 ｜ 累積分數: 0 ｜ ε=0.199



Round 2 你的出拳(r/p/s/空白隨機):  p


你: paper     ｜ AI: rock      ｜ 結果: AI輸 ｜ 累積分數: -1 ｜ ε=0.198


...

你: scissors  ｜ AI: rock      ｜ 結果: AI贏 ｜ 累積分數: 0 ｜ ε=0.195



Round 6 你的出拳(r/p/s/空白隨機):  s


你: scissors  ｜ AI: rock      ｜ 結果: AI贏 ｜ 累積分數: 1 ｜ ε=0.194



Round 7 你的出拳(r/p/s/空白隨機):  p


你: paper     ｜ AI: rock      ｜ 結果: AI輸 ｜ 累積分數: 0 ｜ ε=0.193


這個程式要讓電腦 AI 學習對戰策略。
核心思想是：

AI 根據過去的狀態（上一手出什麼）來選擇動作（AI 出什麼），並根據勝負獲得獎勵，然後更新學習表。

基本核心流程：

1.人類出拳（或隨機代打）

2.AI 用 ε-greedy 策略出拳（有時隨機、有時根據學過的最佳動作）

3.比較勝負 → 得到 reward (+1, 0, -1)

4.更新 Q-table（學習經驗）

5.降低 ε（AI 逐漸變得更「聰明」，少隨機）